# 07 - ROC Curves and AUC

In this notebook, we explore the Receiver Operating Characteristic (ROC) Curve and the Area Under the Curve (AUC).

## Concept Overview
Instead of evaluating a model at a single arbitrary threshold (like 0.5), the ROC curve plots the model's performance across *every possible threshold* from 0.0 to 1.0.

## Visual Demonstration: Building the Curve
We plot the True Positive Rate (Recall) on the Y-axis and the False Positive Rate on the X-axis.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# Generate Data
X, y = make_classification(n_samples=1000, weights=[0.5, 0.5], random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Train Model
model = LogisticRegression()
model.fit(X_train, y_train)

# Get probabilities for Class 1
y_prob = model.predict_proba(X_test)[:, 1]

# Calculate ROC coordinates
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

# Plot
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Guessing')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC)')
plt.legend(loc="lower right")
plt.show()

## Interpretation
* The **Random Guessing** line (AUC = 0.5) is the baseline. A monkey flipping a coin gets an AUC of 0.5.
* The **Top Left Corner** (TPR=1.0, FPR=0.0) is perfection.
* Our model has an AUC of 0.88, meaning there is an 88% chance that our model will rank a randomly chosen positive instance higher than a randomly chosen negative instance.

## Scikit-Learn Implementation: Comparing Models
ROC curves are the best way to visually compare two different models.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Train a second model
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)
rf_prob = rf_model.predict_proba(X_test)[:, 1]

# Get ROC for RF
rf_fpr, rf_tpr, _ = roc_curve(y_test, rf_prob)
rf_auc = auc(rf_fpr, rf_tpr)

# Plot both
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'Logistic Regression (AUC = {roc_auc:.2f})')
plt.plot(rf_fpr, rf_tpr, label=f'Random Forest (AUC = {rf_auc:.2f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Model Comparison via ROC')
plt.legend(loc="lower right")
plt.show()

## Industry Notes
ROC AUC is highly resilient to class imbalance, but it can sometimes present an overly optimistic view of model performance on highly imbalanced datasets. For extreme imbalance, use Precision-Recall Curves (Notebook 08).

## Exercises
1. Find the exact threshold in the `thresholds` array that gives a False Positive Rate closest to `0.10`.
2. Create a perfectly terrible model (AUC < 0.5) by predicting the exact opposite of the Logistic Regression probabilities (`1 - y_prob`). Plot its ROC curve.